# MWE 51 - Presentation-scale MMS replication

This experiment treats the numerical values in the supplied USFEM
presentations and reports as regression targets. A replication passes only
when the case definition, coefficients, finite-element pair, facet law, and
finest mesh pair agree with the stored profile and every compared scalar lies
within its documented tolerance.

The default live run reproduces the two-dimensional equal-order boundary-layer
study. The larger three-dimensional MMS and body-fitted vug runs are opt-in
because they require substantially more memory and solve time.

In [1]:
from __future__ import annotations

import pandas as pd

try:
    from IPython.display import display
except ImportError:  # pragma: no cover - notebook convenience fallback
    display = print

from voids.examples.mms import (
    presentation_mms_references,
    presentation_vug_references,
    run_presentation_mms,
    run_presentation_vug,
)
from voids.fem.singlephase import FEniCSSolverOptions

## Reproducible configuration

The 2D default runs through \(128^2\), matching the equal-order report row.
Set the 3D and vug flags only when report-scale cost is acceptable. The 3D
linear MMS sequence ends at \(20^3\); the vug case regenerates the body-fitted
Gmsh mesh with target size \(\sqrt{3}/30\).

In [2]:
two_dimensional_reference = "2d_boundary_layer_p1dg1"
run_three_dimensional_face3d = False
run_report_scale_vug = False
solver_options = FEniCSSolverOptions.superlu_direct()

## Auditable baseline catalogue

These are tracked numerical targets; no ignored `tmp/` file is read at
runtime. Rates use the final consecutive mesh pair from each supplied study.

In [3]:
mms_catalogue_rows = []
for reference in presentation_mms_references():
    mms_catalogue_rows.append(
        {
            "name": reference.name,
            "dimension": reference.case.dimension,
            "method": reference.method,
            "facet_law": reference.facet_law,
            "facet_size_mode": reference.facet_size_mode,
            "finest_pair": reference.resolutions[-2:],
            "target_metrics": ", ".join(
                quantity.metric for quantity in reference.quantities
            ),
        }
    )
mms_catalogue = pd.DataFrame(mms_catalogue_rows)
display(mms_catalogue)

vug_catalogue = pd.DataFrame(
    [
        {
            "name": reference.name,
            "method": reference.method,
            "resolution": reference.benchmark.resolution,
            "facet_law": reference.facet_law,
            "facet_size_mode": reference.facet_size_mode,
            "target_flow_rate": next(
                quantity.expected
                for quantity in reference.quantities
                if quantity.metric == "flow_rate"
            ),
        }
        for reference in presentation_vug_references()
    ]
)
display(vug_catalogue)

,name,dimension,method,facet_law,facet_size_mode,finest_pair,target_metrics
0,2d_boundary_layer_p1dg0,2,usfem_p1dg0,reaction_diffusion,facet_diameter,"(128, 256)","velocity_l2_error, velocity_h1_error, pressure..."
1,2d_boundary_layer_p1dg1,2,usfem_p1dg1,reaction_diffusion,facet_diameter,"(64, 128)","velocity_l2_error, velocity_h1_error, pressure..."
2,3d_bubble_usfem_p1dg0_classic,3,usfem_p1dg0,classic,representative,"(16, 20)","velocity_l2_rate, velocity_h1_rate, pressure_l..."
3,3d_bubble_usfem_p1dg0_shifted,3,usfem_p1dg0,shifted,representative,"(16, 20)","velocity_l2_rate, velocity_h1_rate, pressure_l..."
4,3d_bubble_usfem_p1dg0_reaction_diffusion,3,usfem_p1dg0,reaction_diffusion,representative,"(16, 20)","velocity_l2_rate, velocity_h1_rate, pressure_l..."
5,3d_bubble_usfem_p1dg0_face3d,3,usfem_p1dg0,face3d,representative,"(16, 20)","velocity_l2_rate, velocity_h1_rate, pressure_l..."
6,3d_bubble_usfem_p1dg1_classic,3,usfem_p1dg1,classic,representative,"(16, 20)","velocity_l2_rate, velocity_h1_rate, pressure_l..."
7,3d_bubble_usfem_p1dg1_shifted,3,usfem_p1dg1,shifted,representative,"(16, 20)","velocity_l2_rate, velocity_h1_rate, pressure_l..."
8,3d_bubble_usfem_p1dg1_reaction_diffusion,3,usfem_p1dg1,reaction_diffusion,representative,"(16, 20)","velocity_l2_rate, velocity_h1_rate, pressure_l..."
9,3d_bubble_usfem_p1dg1_face3d,3,usfem_p1dg1,face3d,representative,"(16, 20)","velocity_l2_rate, velocity_h1_rate, pressure_l..."


,name,method,resolution,facet_law,facet_size_mode,target_flow_rate
0,3d_centered_vug_p1dg1,usfem_p1dg1,30,shifted,facet_measure,2.413000e-07
1,3d_centered_vug_taylor_hood,taylor_hood,30,NaN,NaN,2.421670e-07


## Replicate the 2D presentation row

In [4]:
two_dimensional_run = run_presentation_mms(
    two_dimensional_reference,
    options=solver_options,
)
two_dimensional_run.assert_matches()

two_dimensional_errors = pd.DataFrame(two_dimensional_run.result.as_dicts())
two_dimensional_comparison = pd.DataFrame(two_dimensional_run.comparison.as_dicts())
display(two_dimensional_errors)
display(two_dimensional_comparison)

,resolution,h,num_cells,num_dofs,solve_seconds,velocity_l2_error,velocity_h1_error,pressure_l2_error,divergence_l2,velocity_l2_rate,velocity_h1_rate,pressure_l2_rate,divergence_l2_rate
0,4,0.250000,32,146,0.001285,0.423892,5.956166,0.202063,0.494060,NaN,NaN,NaN,NaN
1,8,0.125000,128,546,0.002724,0.281098,8.421432,0.153747,0.702501,0.592623,-0.499682,0.394250,-0.507816
2,16,0.062500,512,2114,0.013777,0.146097,8.231843,0.081036,0.459128,0.944147,0.032850,0.923920,0.613603
3,32,0.031250,2048,8322,0.108702,0.062281,6.447601,0.042787,0.252550,1.230065,0.352453,0.921386,0.862330
4,64,0.015625,8192,33026,1.622819,0.020334,4.050596,0.022002,0.132658,1.614867,0.670628,0.959538,0.928859
5,128,0.007812,32768,131586,20.789356,0.005577,2.191324,0.010851,0.062464,1.866268,0.886331,1.019827,1.086604


,metric,observed,expected,absolute_error,relative_error,passed
0,velocity_l2_error,0.005577,0.005577,3.622810e-07,0.000065,True
1,velocity_h1_error,2.191324,2.191000,3.244792e-04,0.000148,True
2,pressure_l2_error,0.010851,0.010850,8.728611e-07,0.000080,True
3,divergence_l2,0.062464,0.062460,4.351690e-06,0.000070,True
4,velocity_l2_rate,1.866268,1.866000,2.684407e-04,0.000144,True
5,pressure_l2_rate,1.019827,1.019000,8.273655e-04,0.000812,True


## Optional 3D face-subscale replication

The supplied 3D table reports separate rows for the P1/DG0 and P1/DG1
formulations. Both use the same polynomial exact solution and
`face_refinement=24`; each live run below executes the complete
\((4,6,8,10,12,16,20)^3\) sequence.

In [5]:
three_dimensional_runs = {}
if run_three_dimensional_face3d:
    for reference_name in (
        "3d_bubble_usfem_p1dg0_face3d",
        "3d_bubble_usfem_p1dg1_face3d",
    ):
        print(f"Running {reference_name}")
        replication = run_presentation_mms(
            reference_name,
            options=solver_options,
        )
        replication.assert_matches()
        three_dimensional_runs[reference_name] = replication

if three_dimensional_runs:
    display(
        pd.concat(
            [
                pd.DataFrame(replication.comparison.as_dicts()).assign(
                    reference=reference_name
                )
                for reference_name, replication in three_dimensional_runs.items()
            ],
            ignore_index=True,
        )
    )
else:
    print("3D report-scale runs disabled; set run_three_dimensional_face3d=True.")

3D report-scale runs disabled; set run_three_dimensional_face3d=True.


## Optional report-scale 3D centered-vug replication

This is not an MMS because no exact flow field is available. It is a
like-for-like body-fitted flux benchmark. Each formulation is checked against
its supplied flux, and their ratio is also reported.

In [6]:
vug_runs = {}
if run_report_scale_vug:
    for reference_name in (
        "3d_centered_vug_taylor_hood",
        "3d_centered_vug_p1dg1",
    ):
        print(f"Running {reference_name}")
        replication = run_presentation_vug(
            reference_name,
            options=solver_options,
        )
        replication.assert_matches()
        vug_runs[reference_name] = replication

if vug_runs:
    taylor_hood_flux = vug_runs["3d_centered_vug_taylor_hood"].result.flow_rate
    p1dg1_flux = vug_runs["3d_centered_vug_p1dg1"].result.flow_rate
    display(
        pd.DataFrame(
            [
                {
                    "Taylor-Hood flux": taylor_hood_flux,
                    "P1/DG1 flux": p1dg1_flux,
                    "relative difference": abs(p1dg1_flux - taylor_hood_flux)
                    / abs(taylor_hood_flux),
                }
            ]
        )
    )
else:
    print("3D report-scale vug runs disabled; set run_report_scale_vug=True.")

3D report-scale vug runs disabled; set run_report_scale_vug=True.


## Interpretation and limitations

- The comparisons use the exact reported finest pair; a quick coarse-grid
  trend cannot pass as a presentation replication.
- Gmsh cell counts may vary slightly with Gmsh version. The vug gate therefore
  checks the represented volume and flux rather than an exact tetrahedron
  count.
- The shipped gates cover raw MMS errors/rates and the centered-vug fluxes
  available from the current `voids` FEM result objects.
- The presentation's Raviart--Thomas recovered-velocity divergence is not
  claimed here because conservative recovery is not yet part of the shipped
  `voids` FEM API. Adding it requires a separate implementation and regression
  layer; raw divergence and recovered divergence must not be conflated.